# Where are the promos?

## Exploración previa

In [0]:
tbl = "tacobueno.tickets.ventas_clean"

In [0]:
# 1. Valores distintos de row_type, conteo y porcentaje
print("=== 1. row_type: conteo y porcentaje ===")
df1 = spark.sql(f"""
    SELECT row_type,
           COUNT(*)          AS filas,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {tbl}
    GROUP BY row_type
    ORDER BY filas DESC
""")
display(df1)

In [0]:
# 2. Valores distintos de transaction_type, conteo y porcentaje
print("=== 2. transaction_type: conteo y porcentaje ===")
df2 = spark.sql(f"""
    SELECT transaction_type,
           COUNT(*)          AS filas,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {tbl}
    GROUP BY transaction_type
    ORDER BY filas DESC
""")
display(df2)

In [0]:
# 3. Cruce transaction_type x row_type, conteo de filas
print("=== 3. Cruce transaction_type x row_type ===")
df3 = spark.sql(f"""
    SELECT transaction_type,
           row_type,
           COUNT(*) AS filas
    FROM {tbl}
    GROUP BY transaction_type, row_type
    ORDER BY transaction_type, row_type
""")
display(df3)

In [0]:
# 4. Tipo de dato de columnas clave
print("=== 4. Tipos de dato de columnas ===")
cols_to_check = [
    "fecha", "begin_time", "end_time", "transaction_ts",
    "hora", "dia_semana", "quantity", "menu_price",
    "item_discount", "total", "total_discount", "tax",
    "pop", "is_partial_day", "is_low_coverage_store"
]
schema_fields = {f.name: f.dataType.simpleString() for f in spark.table(tbl).schema.fields}
rows = []
for c in cols_to_check:
    rows.append((c, schema_fields.get(c, "(no existe)")))
df4 = spark.createDataFrame(rows, ["columna", "tipo_de_dato"])
display(df4)

In [0]:
# 5. Totales: filas, tickets distintos, rango de fecha
print("=== 5. Totales generales ===")
df5 = spark.sql(f"""
    SELECT COUNT(*)                                                       AS total_filas,
           COUNT(DISTINCT CONCAT_WS('|', store_id, CAST(fecha AS STRING), check_number)) AS tickets_distintos,
           MIN(fecha)                                                     AS fecha_min,
           MAX(fecha)                                                     AS fecha_max
    FROM {tbl}
""")
display(df5)

In [0]:
# 6. Por mes (anio_mes): filas, tickets distintos, suma de item_discount
print("=== 6. Resumen mensual ===")
df6 = spark.sql(f"""
    SELECT anio_mes,
           COUNT(*)                                                       AS filas,
           COUNT(DISTINCT CONCAT_WS('|', store_id, CAST(fecha AS STRING), check_number)) AS tickets_distintos,
           SUM(item_discount)                                             AS suma_item_discount
    FROM {tbl}
    GROUP BY anio_mes
    ORDER BY anio_mes
""")
display(df6)

## Exploración previa 2

In [0]:
tbl = "tacobueno.discounts.ventas_clean"

ticket_id = "CONCAT_WS('|', store_id, CAST(fecha AS STRING), check_number)"

# ─────────────────────────────────────────────────────────────
# 1. Integridad del ticket
# ─────────────────────────────────────────────────────────────

# 1a. Filas con nulo en store_id, fecha o check_number
print("=== 1a. Filas con nulo en store_id, fecha o check_number ===")
df_1a = spark.sql(f"""
    SELECT COUNT(*) AS filas_con_nulo
    FROM {tbl}
    WHERE store_id IS NULL
       OR fecha IS NULL
       OR check_number IS NULL
""")
display(df_1a)

# 1b. Tickets con más de un valor distinto en total, total_discount, tax, pop, begin_time
print("=== 1b. Tickets con >1 valor distinto por campo ===")
df_1b = spark.sql(f"""
    WITH ticket_vals AS (
        SELECT {ticket_id} AS ticket,
               COUNT(DISTINCT total)          AS n_total,
               COUNT(DISTINCT total_discount) AS n_total_discount,
               COUNT(DISTINCT tax)            AS n_tax,
               COUNT(DISTINCT pop)            AS n_pop,
               COUNT(DISTINCT begin_time)     AS n_begin_time
        FROM {tbl}
        GROUP BY {ticket_id}
    )
    SELECT SUM(CASE WHEN n_total          > 1 THEN 1 ELSE 0 END) AS tickets_mult_total,
           SUM(CASE WHEN n_total_discount > 1 THEN 1 ELSE 0 END) AS tickets_mult_total_discount,
           SUM(CASE WHEN n_tax            > 1 THEN 1 ELSE 0 END) AS tickets_mult_tax,
           SUM(CASE WHEN n_pop            > 1 THEN 1 ELSE 0 END) AS tickets_mult_pop,
           SUM(CASE WHEN n_begin_time     > 1 THEN 1 ELSE 0 END) AS tickets_mult_begin_time,
           COUNT(*)                                          AS tickets_totales
    FROM ticket_vals
""")
display(df_1b)

# 1c. Tickets con nulos mezclados con valores en total_discount
print("=== 1c. Tickets con nulos y valores mezclados en total_discount ===")
df_1c = spark.sql(f"""
    WITH ticket_flags AS (
        SELECT {ticket_id} AS ticket,
               MAX(CASE WHEN total_discount IS NULL THEN 1 ELSE 0 END) AS tiene_nulo,
               MAX(CASE WHEN total_discount IS NOT NULL THEN 1 ELSE 0 END) AS tiene_valor
        FROM {tbl}
        GROUP BY {ticket_id}
    )
    SELECT COUNT(*) AS tickets_nulos_y_valores
    FROM ticket_flags
    WHERE tiene_nulo = 1 AND tiene_valor = 1
""")
display(df_1c)

# ─────────────────────────────────────────────────────────────
# 2. Por anio_mes y row_type
# ─────────────────────────────────────────────────────────────
print("=== 2. Resumen por anio_mes y row_type ===")
df2 = spark.sql(f"""
    SELECT anio_mes,
           row_type,
           COUNT(*)                                              AS filas,
           SUM(CASE WHEN menu_price != 0 THEN 1 ELSE 0 END)     AS filas_menu_price_no_cero,
           SUM(CASE WHEN menu_price < 0  THEN 1 ELSE 0 END)     AS filas_menu_price_negativo,
           SUM(quantity * menu_price)                           AS suma_qty_x_menu_price
    FROM {tbl}
    GROUP BY anio_mes, row_type
    ORDER BY anio_mes, row_type
""")
display(df2)

# ─────────────────────────────────────────────────────────────
# 3. Perfil de líneas Discount
# ─────────────────────────────────────────────────────────────

# 3a. Por row_type dentro de transaction_type = 'Discount'
print("=== 3a. Perfil de row_type dentro de transaction_type = 'Discount' ===")
df_3a = spark.sql(f"""
    SELECT row_type,
           COUNT(*)                              AS filas,
           MIN(quantity)                          AS min_quantity,
           ROUND(AVG(quantity), 4)                AS avg_quantity,
           MAX(quantity)                          AS max_quantity,
           MIN(menu_price)                        AS min_menu_price,
           ROUND(AVG(menu_price), 4)               AS avg_menu_price,
           MAX(menu_price)                        AS max_menu_price,
           MIN(item_discount)                     AS min_item_discount,
           ROUND(AVG(item_discount), 4)            AS avg_item_discount,
           MAX(item_discount)                     AS max_item_discount,
           SUM(CASE WHEN item_discount = 0 THEN 1 ELSE 0 END) AS filas_item_discount_cero
    FROM {tbl}
    WHERE transaction_type = 'Discount'
    GROUP BY row_type
    ORDER BY filas DESC
""")
display(df_3a)

# 3b. Filas con transaction_type = 'Discount' por anio_mes
print("=== 3b. Filas Discount por anio_mes ===")
df_3b = spark.sql(f"""
    SELECT anio_mes,
           COUNT(*) AS filas_discount
    FROM {tbl}
    WHERE transaction_type = 'Discount'
    GROUP BY anio_mes
    ORDER BY anio_mes
""")
display(df_3b)

# ─────────────────────────────────────────────────────────────
# 4. Curva mensual del descuento a nivel ticket
# ─────────────────────────────────────────────────────────────
print("=== 4. Curva mensual del descuento (nivel ticket) ===")
df4 = spark.sql(f"""
    WITH ticket_agg AS (
        SELECT {ticket_id}        AS ticket,
               anio_mes,
               MAX(total_discount) AS total_discount,
               SUM(CASE WHEN transaction_type = 'Discount' THEN 1 ELSE 0 END) AS lineas_discount
        FROM {tbl}
        GROUP BY {ticket_id}, anio_mes
    )
    SELECT anio_mes,
           COUNT(*)                                                                     AS tickets_totales,
           SUM(CASE WHEN total_discount > 0 THEN 1 ELSE 0 END)                          AS tickets_con_descuento,
           ROUND(SUM(CASE WHEN total_discount > 0 THEN 1 ELSE 0 END) * 100.0
                 / COUNT(*), 2)                                                         AS pct_con_descuento,
           SUM(CASE WHEN total_discount > 0 AND lineas_discount = 0 THEN 1 ELSE 0 END) AS tickets_descuento_sin_lineas_discount,
           ROUND(AVG(CASE WHEN total_discount > 0 THEN total_discount END), 4)          AS promedio_total_discount_positivo
    FROM ticket_agg
    GROUP BY anio_mes
    ORDER BY anio_mes
""")
display(df4)

# ─────────────────────────────────────────────────────────────
# 5. Muestras para inspección visual
# ─────────────────────────────────────────────────────────────

cols = """store_id, fecha, check_number, transaction_type, row_type,
         menu_item, pos_code, quantity, menu_price, item_discount,
         total, total_discount, tax, transaction_ts"""

# 5a. 3 tickets de 2025-06 con al menos una línea Discount (máx 12 líneas)
print("=== 5a. 3 tickets 2025-06 con línea Discount ===")
df_5a = spark.sql(f"""
    WITH ticket_lines AS (
        SELECT {ticket_id} AS ticket
        FROM {tbl}
        WHERE anio_mes = '2025-06'
          AND transaction_type = 'Discount'
        GROUP BY {ticket_id}
        HAVING COUNT(*) <= 12
    ),
    picked AS (
        SELECT ticket
        FROM ticket_lines
        ORDER BY ticket
        LIMIT 3
    )
    SELECT {cols}
    FROM {tbl}
    WHERE anio_mes = '2025-06'
      AND {ticket_id} IN (SELECT ticket FROM picked)
    ORDER BY transaction_ts
""")
display(df_5a)

# 5b. 3 tickets de 2026-03 con total_discount > 0 (máx 12 líneas)
print("=== 5b. 3 tickets 2026-03 con total_discount > 0 ===")
df_5b = spark.sql(f"""
    WITH ticket_lines AS (
        SELECT {ticket_id} AS ticket
        FROM {tbl}
        WHERE anio_mes = '2026-03'
        GROUP BY {ticket_id}
        HAVING MAX(total_discount) > 0 AND COUNT(*) <= 12
    ),
    picked AS (
        SELECT ticket
        FROM ticket_lines
        ORDER BY ticket
        LIMIT 3
    )
    SELECT {cols}
    FROM {tbl}
    WHERE anio_mes = '2026-03'
      AND {ticket_id} IN (SELECT ticket FROM picked)
    ORDER BY transaction_ts
""")
display(df_5b)

# 5c. 3 tickets de 2026-03 con total_discount = 0 y al menos una línea price_adjustment (máx 12 líneas)
print("=== 5c. 3 tickets 2026-03 con total_discount = 0 y línea price_adjustment ===")
df_5c = spark.sql(f"""
    WITH ticket_lines AS (
        SELECT {ticket_id} AS ticket
        FROM {tbl}
        WHERE anio_mes = '2026-03'
        GROUP BY {ticket_id}
        HAVING MAX(total_discount) = 0
           AND MAX(CASE WHEN row_type = 'price_adjustment' THEN 1 ELSE 0 END) = 1
           AND COUNT(*) <= 12
    ),
    picked AS (
        SELECT ticket
        FROM ticket_lines
        ORDER BY ticket
        LIMIT 3
    )
    SELECT {cols}
    FROM {tbl}
    WHERE anio_mes = '2026-03'
      AND {ticket_id} IN (SELECT ticket FROM picked)
    ORDER BY transaction_ts
""")
display(df_5c)


| # | Hallazgo | Implicación |
|---|---|---|
| 1 | Los tickets distintos (13,788,596) coinciden con bronze. No hay nulos en la llave `store_id + fecha + check_number`. | La limpieza no perdió tickets y la llave es confiable. |
| 2 | Los campos de ticket casi no varían dentro del ticket. Tienen más de un valor: `total` 71 tickets, `total_discount` 9, `tax` 72, `begin_time` 72, `pop` 58 (≈0.0005%). | `MAX` es válido para campos de ticket. Estos tickets se marcan y se excluyen del cluster. |
| 3 | `item_discount` suma exactamente 0 desde 2026-02. Antes baja gradualmente: ~0.3 por ticket en 2025, 95,889 en dic 2025 y 18,227 en ene 2026. | P3 empieza en feb 2026 y dic 2025 a ene 2026 es transición (P2). |
| 4 | Las líneas `Discount` (1,619,497) son la propia línea de producto, con su `quantity` y `menu_price`. Desaparecen desde feb 2026 (58,494 en dic 2025, 9,703 en ene 2026, 0 después). | `SUM(quantity)` sobre `row_type = 'product'` no duplica artículos. |
| 5 | El % de tickets con `total_discount > 0` es continuo: 7.5 a 10.9% (abr a nov 2025), 6.6% (dic), 6.4% (ene), 6.5 a 7.9% (feb a sep 2026). No hay salto hacia 45%. | La promoción no está oculta: cambió de mecanismo de registro. `total_discount` es candidato a medida comparable entre periodos. |
| 6 | Tickets con descuento y sin línea `Discount`: 834 (nov 2025), 12,445 (dic), 31,966 (ene) y 100% desde feb 2026. | El cambio de registro empieza en nov 2025. Desde feb 2026, `total_discount` es la única señal a nivel ticket. |
| 7 | Las líneas `price_adjustment` (`menu_item = 'Item Promo'`, precio negativo, sufijo como `(Coupon)`) pasan de 12 en todo 2025 a 1,451 (dic), 10,840 (ene) y un pico de 58,904 (may 2026), y bajan a ~17 a 19 mil en ago y sep. | Es un segundo mecanismo de promoción. Cubre como máximo ~8% de los tickets por mes, así que ninguno de los dos explica un 45%. |
| 8 | `SUM(quantity * menu_price)` de modificadores es 0 en todos los meses. Desde feb 2026 ningún modificador trae precio. | Los modificadores se excluyen del monto y del conteo de artículos. |
| 9 | En 6 tickets de la tienda 5026, `total` equivale al bruto en 2025 y a lo cobrado en 2026 (productos + donación + impuesto − descuento − ajustes). | Hipótesis: `total` cambió de definición, y parte del 45% sin match sería ese cambio y no descuentos ocultos. Pendiente de las ecuaciones E1 a E5. |
| 10 | En 2025 una línea puede agrupar cantidad (qty 4). En 2026 el mismo producto se repite en líneas con qty 1. | El conteo de líneas no es comparable entre años. Usar `SUM(quantity)` de productos. |
| 11 | La intensidad de promoción en P1 no es homogénea: 21.3%, 18.9% y 14.5% de tickets con descuento en ene a mar 2025, contra ~8% de abr a nov. | P1 necesita control por mes o separar ene a mar para que el cluster no refleje solo el lanzamiento. |
| 12 | `check_number` pasa de 6 dígitos (2025) a 14 (2026). Desde feb 2026 desaparecen las filas `void` y casi todas las `delivery_fee`. | Probable cambio de POS. Limita la comparabilidad entre periodos y es pregunta para el cliente. |

## Exploratorio 3

#### Fase 1

In [0]:
tbl = "tacobueno.discounts.ventas_clean"
ticket_id = "CONCAT_WS('|', store_id, CAST(fecha AS STRING), check_number)"

# ─────────────────────────────────────────────────────────────
# CTE base: agregación por ticket
# ─────────────────────────────────────────────────────────────
base_cte = f"""
WITH ticket_agg AS (
    SELECT {ticket_id}        AS ticket,
           store_id,
           fecha,
           check_number,
           anio_mes,
           MAX(total)          AS total,
           MAX(total_discount) AS total_discount,
           MAX(tax)            AS tax,
           MAX(pop)            AS pop,
           MAX(begin_time)     AS begin_time,
           SUM(CASE WHEN row_type = 'product' THEN quantity * menu_price ELSE 0 END) AS sub_prod,
           SUM(CASE WHEN row_type IN ('donation','delivery_fee','tip') THEN quantity * menu_price ELSE 0 END) AS sub_otros,
           SUM(CASE WHEN row_type = 'price_adjustment' THEN quantity * menu_price ELSE 0 END) AS adj,
           SUM(CASE WHEN row_type = 'product' THEN item_discount ELSE 0 END) AS item_disc_prod,
           SUM(item_discount) AS item_disc_all,
           SUM(CASE WHEN row_type = 'price_adjustment' THEN 1 ELSE 0 END) AS n_adj,
           SUM(CASE WHEN transaction_type = 'Discount' THEN 1 ELSE 0 END) AS n_lineas_discount,
           SUM(CASE WHEN row_type = 'void' THEN 1 ELSE 0 END) AS n_void,
           SUM(CASE WHEN row_type = 'delivery_fee' THEN 1 ELSE 0 END) AS n_delivery_fee,
           COUNT(*)            AS n_lineas,
           MAX(CASE WHEN row_type = 'product' THEN quantity END) AS max_qty,
           COUNT(DISTINCT total)          AS n_distintos_total,
           COUNT(DISTINCT tax)            AS n_distintos_tax,
           COUNT(DISTINCT pop)            AS n_distintos_pop,
           COUNT(DISTINCT begin_time)     AS n_distintos_begin_time,
           COUNT(DISTINCT total_discount) AS n_distintos_total_discount,
           CASE WHEN COUNT(DISTINCT total) > 1
                  OR COUNT(DISTINCT tax) > 1
                  OR COUNT(DISTINCT pop) > 1
                  OR COUNT(DISTINCT begin_time) > 1
                  OR COUNT(DISTINCT total_discount) > 1
                THEN 1 ELSE 0 END AS flag_inconsistente
    FROM {tbl}
    GROUP BY {ticket_id}, store_id, fecha, check_number, anio_mes
)
"""

# ─────────────────────────────────────────────────────────────
# A1. total_discount vs item_disc_prod por mes (2025-01 a 2026-01)
# ─────────────────────────────────────────────────────────────
print("=== A1. total_discount vs item_disc_prod por mes ===")
df_a1 = spark.sql(base_cte + """
SELECT anio_mes,
       COUNT(*) AS tickets,
       ROUND(SUM(CASE WHEN total_discount = 0 AND item_disc_prod = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_td0_id0,
       ROUND(SUM(CASE WHEN total_discount > 0 AND item_disc_prod > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_td_pos_id_pos,
       ROUND(SUM(CASE WHEN total_discount > 0 AND item_disc_prod = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_td_pos_id0,
       ROUND(SUM(CASE WHEN total_discount = 0 AND item_disc_prod > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_td0_id_pos
FROM ticket_agg
WHERE anio_mes BETWEEN '2025-01' AND '2026-01'
GROUP BY anio_mes
ORDER BY anio_mes
""")
display(df_a1)

In [0]:
# ─────────────────────────────────────────────────────────────
# 2. Traslape de mecanismos en 2026
# ─────────────────────────────────────────────────────────────
print("=== 2. Traslape de mecanismos (desde 2025-11) ===")
df2 = spark.sql(base_cte + """
SELECT anio_mes,
       COUNT(*) AS tickets,
       SUM(CASE WHEN total_discount > 0 AND n_adj = 0 THEN 1 ELSE 0 END) AS solo_total_discount,
       SUM(CASE WHEN total_discount = 0 AND n_adj > 0 THEN 1 ELSE 0 END) AS solo_adj,
       SUM(CASE WHEN total_discount > 0 AND n_adj > 0 THEN 1 ELSE 0 END) AS ambos,
       SUM(CASE WHEN total_discount = 0 AND n_adj = 0 THEN 1 ELSE 0 END) AS ninguno,
       ROUND(SUM(CASE WHEN total_discount > 0 AND n_adj = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_solo_td,
       ROUND(SUM(CASE WHEN total_discount = 0 AND n_adj > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_solo_adj,
       ROUND(SUM(CASE WHEN total_discount > 0 AND n_adj > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_ambos,
       ROUND(SUM(CASE WHEN total_discount = 0 AND n_adj = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_ninguno
FROM ticket_agg
WHERE anio_mes >= '2025-11'
GROUP BY anio_mes
ORDER BY anio_mes
""")
display(df2)

In [0]:
# ─────────────────────────────────────────────────────────────
# 3a. Perfil de price_adjustment por sufijo de pos_code
# ─────────────────────────────────────────────────────────────
print("=== 3a. Perfil de price_adjustment por sufijo de pos_code ===")
df3a = spark.sql(f"""
SELECT anio_mes,
       CASE WHEN regexp_extract(pos_code, '[(]([^)]+)[)]', 1) = ''
            THEN 'sin_sufijo'
            ELSE regexp_extract(pos_code, '[(]([^)]+)[)]', 1)
       END AS sufijo,
       COUNT(*) AS lineas,
       COUNT(DISTINCT {ticket_id}) AS tickets_distintos,
       ROUND(AVG(quantity * menu_price), 4) AS prom_qty_x_price,
       SUM(quantity * menu_price) AS suma_qty_x_price
FROM {tbl}
WHERE row_type = 'price_adjustment'
  AND anio_mes >= '2025-11'
GROUP BY anio_mes,
         CASE WHEN regexp_extract(pos_code, '[(]([^)]+)[)]', 1) = ''
              THEN 'sin_sufijo'
              ELSE regexp_extract(pos_code, '[(]([^)]+)[)]', 1)
         END
ORDER BY anio_mes, sufijo
""")
display(df3a)

In [0]:
# ─────────────────────────────────────────────────────────────
# 3b. Top 10 menu_item más frecuentes en price_adjustment
# ─────────────────────────────────────────────────────────────
print("=== 3b. Top 10 menu_item en price_adjustment ===")
df3b = spark.sql(f"""
SELECT menu_item,
       COUNT(*) AS lineas
FROM {tbl}
WHERE row_type = 'price_adjustment'
GROUP BY menu_item
ORDER BY lineas DESC
LIMIT 10
""")
display(df3b)

In [0]:
# ─────────────────────────────────────────────────────────────
# 4. Razón de descuento: percentiles por periodo
# ─────────────────────────────────────────────────────────────
print("=== 4. Razón de descuento (total_discount / sub_prod) ===")
df4 = spark.sql(base_cte + """
SELECT CASE WHEN anio_mes BETWEEN '2025-04' AND '2025-11' THEN '2025-04 a 2025-11'
            WHEN anio_mes BETWEEN '2026-02' AND '2026-09' THEN '2026-02 a 2026-09'
       END AS periodo,
       COUNT(*) AS tickets,
       ROUND(percentile_approx(total_discount / sub_prod, 0.10), 4) AS p10,
       ROUND(percentile_approx(total_discount / sub_prod, 0.25), 4) AS p25,
       ROUND(percentile_approx(total_discount / sub_prod, 0.50), 4) AS p50,
       ROUND(percentile_approx(total_discount / sub_prod, 0.75), 4) AS p75,
       ROUND(percentile_approx(total_discount / sub_prod, 0.90), 4) AS p90,
       ROUND(SUM(CASE WHEN ABS(total_discount / sub_prod - 0.10) <= 0.01 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_cerca_010,
       ROUND(SUM(CASE WHEN ABS(total_discount / sub_prod - 0.20) <= 0.01 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_cerca_020,
       ROUND(SUM(CASE WHEN ABS(total_discount / sub_prod - 1.00) <= 0.01 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_cerca_100
FROM ticket_agg
WHERE total_discount > 0 AND sub_prod > 0
  AND (anio_mes BETWEEN '2025-04' AND '2025-11' OR anio_mes BETWEEN '2026-02' AND '2026-09')
GROUP BY CASE WHEN anio_mes BETWEEN '2025-04' AND '2025-11' THEN '2025-04 a 2025-11'
             WHEN anio_mes BETWEEN '2026-02' AND '2026-09' THEN '2026-02 a 2026-09'
        END
ORDER BY periodo
""")
display(df4)

In [0]:
# ─────────────────────────────────────────────────────────────
# 5. Formato de check_number: longitud por mes
# ─────────────────────────────────────────────────────────────
print("=== 5. Formato de check_number por mes ===")
df5 = spark.sql(f"""
WITH len_counts AS (
    SELECT anio_mes,
           LENGTH(check_number) AS len_cn,
           COUNT(*) AS cnt
    FROM {tbl}
    GROUP BY anio_mes, LENGTH(check_number)
),
mode_len AS (
    SELECT anio_mes, len_cn AS mode_len
    FROM (
        SELECT anio_mes, len_cn, cnt,
               ROW_NUMBER() OVER (PARTITION BY anio_mes ORDER BY cnt DESC, len_cn) AS rn
        FROM len_counts
    )
    WHERE rn = 1
),
basic_stats AS (
    SELECT anio_mes,
           MIN(LENGTH(check_number)) AS min_len,
           MAX(LENGTH(check_number)) AS max_len,
           COUNT(DISTINCT {ticket_id}) AS check_numbers_distintos
    FROM {tbl}
    GROUP BY anio_mes
)
SELECT b.anio_mes,
       b.min_len,
       b.max_len,
       m.mode_len,
       b.check_numbers_distintos
FROM basic_stats b
JOIN mode_len m ON b.anio_mes = m.anio_mes
ORDER BY b.anio_mes
""")
display(df5)

#### Resumen de hallazgos

1. Ecuaciones por mes: Cambió la forma de medirlo ~2025-11/2026-01:

    - En 2025 domina E1 (total = sub_prod + sub_otros + adj, sin tax ni descuento) con ~99.8% de cumplimiento.
    - Desde 2026-01 domina E3 (total = sub_prod + sub_otros + adj + tax - total_discount) con ~99%.
    - Solo ~1% de tickets no cumple ninguna ecuación.
2. Traslape de mecanismos — ~90% de tickets no usa ningún mecanismo. total_discount solo: ~6-8%. price_adjustment solo: crece de 0% (2025-11) a 2.7% (2026-04). Ambos simultáneamente: <0.3%.

3. Perfil de price_adjustment — Tres sufijos principales: Coupon (descuento fijo ~-$2.7-3.3), OrderReduction (ajuste variable ~-$0.7-1.1), Bogo (2x1, ~-$1.3-4.5). Desde 2026-07 el pos_code pierde paréntesis → sin_sufijo. El menu_item dominante es "Item Promo" (219K líneas).

4. Razón de descuento — La mediana total_discount/sub_prod bajó de 0.148 (2025) a 0.100 (2026). El % de tickets con razón ≈0.10 subió de 43.9% a 58.8%. Los descuentos ≈100% (artículos gratis) bajaron de 22% a 14%.

5. Formato de check_number — Longitud uniforme de 6 caracteres (2025-01 a 2025-10), transición (2025-11 a 2026-01), luego 14 caracteres (2026-02+). Confirma un cambio de sistema POS en la frontera 2025→2026.


In [0]:
# ─────────────────────────────────────────────────────────────
# A2. Coincidencia y razón total_discount / item_disc_prod (td>0, id>0)
# ─────────────────────────────────────────────────────────────
print("=== A2. Coincidencia y razón td/item_disc_prod (2025-01 a 2026-01) ===")
df_a2 = spark.sql(base_cte + """
SELECT anio_mes,
       COUNT(*) AS tickets,
       ROUND(SUM(CASE WHEN ABS(total_discount - item_disc_prod) < 0.015 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_coinc_prod,
       ROUND(SUM(CASE WHEN ABS(total_discount - item_disc_all) < 0.015 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_coinc_all,
       ROUND(percentile_approx(total_discount / item_disc_prod, 0.10), 4) AS p10,
       ROUND(percentile_approx(total_discount / item_disc_prod, 0.25), 4) AS p25,
       ROUND(percentile_approx(total_discount / item_disc_prod, 0.50), 4) AS p50,
       ROUND(percentile_approx(total_discount / item_disc_prod, 0.75), 4) AS p75,
       ROUND(percentile_approx(total_discount / item_disc_prod, 0.90), 4) AS p90
FROM ticket_agg
WHERE total_discount > 0 AND item_disc_prod > 0
  AND anio_mes BETWEEN '2025-01' AND '2026-01'
GROUP BY anio_mes
ORDER BY anio_mes
""")
display(df_a2)

In [0]:
# ─────────────────────────────────────────────────────────────
# B1. Tickets sin_ecuacion por mes
# ─────────────────────────────────────────────────────────────
print("=== B1. Tickets sin_ecuacion por mes ===")
df_b1 = spark.sql(base_cte + """
SELECT anio_mes,
       COUNT(*) AS tickets,
       SUM(sin_ecuacion) AS n_sin_ecuacion,
       ROUND(SUM(sin_ecuacion) * 100.0 / COUNT(*), 2) AS pct_sin_ecuacion,
       ROUND(SUM(CASE WHEN sin_ecuacion = 1 AND flag_inconsistente = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(sin_ecuacion), 0), 2) AS pct_inconsistente,
       ROUND(SUM(CASE WHEN sin_ecuacion = 1 AND n_void > 0 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(sin_ecuacion), 0), 2) AS pct_con_void,
       ROUND(SUM(CASE WHEN sin_ecuacion = 1 AND n_delivery_fee > 0 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(sin_ecuacion), 0), 2) AS pct_con_delivery,
       ROUND(SUM(CASE WHEN sin_ecuacion = 1 AND n_lineas_discount > 0 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(sin_ecuacion), 0), 2) AS pct_con_lineas_disc,
       ROUND(SUM(CASE WHEN sin_ecuacion = 1 AND n_adj > 0 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(sin_ecuacion), 0), 2) AS pct_con_adj,
       ROUND(SUM(CASE WHEN sin_ecuacion = 1 AND max_qty >= 10 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(sin_ecuacion), 0), 2) AS pct_max_qty_ge_10,
       ROUND(SUM(CASE WHEN sin_ecuacion = 1 AND total_discount > 0 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(sin_ecuacion), 0), 2) AS pct_con_td_pos
FROM (
    SELECT anio_mes, flag_inconsistente, n_void, n_delivery_fee,
           n_lineas_discount, n_adj, max_qty, total_discount,
           CASE WHEN ABS(total - (sub_prod + sub_otros + adj)) < 0.015 THEN 0
                 WHEN ABS(total - (sub_prod + sub_otros + adj + tax)) < 0.015 THEN 0
                 WHEN ABS(total - (sub_prod + sub_otros + adj + tax - total_discount)) < 0.015 THEN 0
                 WHEN ABS(total - (sub_prod + sub_otros + adj - total_discount)) < 0.015 THEN 0
                 WHEN ABS(total - (sub_prod + sub_otros - item_disc_prod)) < 0.015 THEN 0
                 ELSE 1 END AS sin_ecuacion
    FROM ticket_agg
)
GROUP BY anio_mes
ORDER BY anio_mes
""")
display(df_b1)

In [0]:
# ─────────────────────────────────────────────────────────────
# B3. Distribución de d en sin_ecuacion (flag_inconsistente=0)
# 2025: d = total - E1 ; 2026: d = total - E3
# ─────────────────────────────────────────────────────────────
print("=== B3. Distribución de d en sin_ecuacion (flag=0) ===")
df_b3 = spark.sql(base_cte + """
SELECT CASE WHEN anio_mes < '2025-11' THEN '2025 (d=total-E1)'
            WHEN anio_mes >= '2026-02' THEN '2026 (d=total-E3)'
       END AS periodo,
       COUNT(*) AS tickets,
       ROUND(percentile_approx(d, 0.01), 4) AS p1,
       ROUND(percentile_approx(d, 0.05), 4) AS p5,
       ROUND(percentile_approx(d, 0.25), 4) AS p25,
       ROUND(percentile_approx(d, 0.50), 4) AS p50,
       ROUND(percentile_approx(d, 0.75), 4) AS p75,
       ROUND(percentile_approx(d, 0.95), 4) AS p95,
       ROUND(percentile_approx(d, 0.99), 4) AS p99,
       ROUND(SUM(CASE WHEN d > 0.015 AND d < 1.00 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_d_pos_chica,
       ROUND(SUM(CASE WHEN d > -1.00 AND d < -0.015 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_d_neg_chica,
       ROUND(SUM(CASE WHEN ABS(d) > 1.00 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_abs_d_grande
FROM (
    SELECT anio_mes,
           CASE WHEN anio_mes < '2025-11' THEN total - (sub_prod + sub_otros + adj)
                WHEN anio_mes >= '2026-02' THEN total - (sub_prod + sub_otros + adj + tax - total_discount)
           END AS d
    FROM ticket_agg
    WHERE flag_inconsistente = 0
      AND ABS(total - (sub_prod + sub_otros + adj)) >= 0.015
      AND ABS(total - (sub_prod + sub_otros + adj + tax)) >= 0.015
      AND ABS(total - (sub_prod + sub_otros + adj + tax - total_discount)) >= 0.015
      AND ABS(total - (sub_prod + sub_otros + adj - total_discount)) >= 0.015
      AND ABS(total - (sub_prod + sub_otros - item_disc_prod)) >= 0.015
      AND (anio_mes < '2025-11' OR anio_mes >= '2026-02')
)
GROUP BY CASE WHEN anio_mes < '2025-11' THEN '2025 (d=total-E1)'
             WHEN anio_mes >= '2026-02' THEN '2026 (d=total-E3)'
        END
ORDER BY periodo
""")
display(df_b3)

In [0]:
# ─────────────────────────────────────────────────────────────
# A3. Tickets td>0, id>0 que NO coinciden (2025-04 a 2025-10)
# ─────────────────────────────────────────────────────────────
print("=== A3a. 10 tickets td>0, id>0, no coinciden ===")
df_a3a = spark.sql(base_cte + """
SELECT ticket, store_id, fecha, check_number,
       total_discount, item_disc_prod, item_disc_all,
       sub_prod, n_lineas_discount, n_lineas
FROM ticket_agg
WHERE total_discount > 0 AND item_disc_prod > 0
  AND ABS(total_discount - item_disc_prod) >= 0.015
  AND anio_mes BETWEEN '2025-04' AND '2025-10'
  AND n_lineas <= 12
ORDER BY anio_mes, store_id, fecha, check_number
LIMIT 10
""")
display(df_a3a)

print("=== A3b. Líneas de esos tickets ===")
tickets_a3 = [r.ticket for r in df_a3a.collect()]
df_a3b = spark.sql(f"""
SELECT store_id, fecha, check_number, transaction_type, row_type,
       menu_item, quantity, menu_price, item_discount
FROM {tbl}
WHERE {ticket_id} IN ({','.join([f"'{t}'" for t in tickets_a3])})
ORDER BY store_id, fecha, check_number, transaction_ts
""")
display(df_a3b)

In [0]:
# ─────────────────────────────────────────────────────────────
# B2. sin_ecuacion por pop (todos los meses)
# ─────────────────────────────────────────────────────────────
print("=== B2. sin_ecuacion por pop ===")
df_b2 = spark.sql(base_cte + """
SELECT pop,
       COUNT(*) AS tickets,
       SUM(sin_ecuacion) AS n_sin_ecuacion,
       ROUND(SUM(sin_ecuacion) * 100.0 / COUNT(*), 2) AS pct_sin_ecuacion
FROM (
    SELECT pop,
           CASE WHEN ABS(total - (sub_prod + sub_otros + adj)) < 0.015 THEN 0
                 WHEN ABS(total - (sub_prod + sub_otros + adj + tax)) < 0.015 THEN 0
                 WHEN ABS(total - (sub_prod + sub_otros + adj + tax - total_discount)) < 0.015 THEN 0
                 WHEN ABS(total - (sub_prod + sub_otros + adj - total_discount)) < 0.015 THEN 0
                 WHEN ABS(total - (sub_prod + sub_otros - item_disc_prod)) < 0.015 THEN 0
                 ELSE 1 END AS sin_ecuacion
    FROM ticket_agg
)
GROUP BY pop
ORDER BY pct_sin_ecuacion DESC
""")
display(df_b2)

In [0]:
# ─────────────────────────────────────────────────────────────
# B4. Inspección visual: sin_ecuacion con flag=0 (5 de 2025, 5 de 2026)
# ─────────────────────────────────────────────────────────────
print("=== B4a. 5 tickets sin_ecuacion flag=0 de 2025 ===")
df_b4a = spark.sql(base_cte + """
SELECT ticket, store_id, fecha, check_number,
       total, sub_prod, sub_otros, adj, tax, total_discount,
       total - (sub_prod + sub_otros + adj) AS d,
       n_lineas
FROM ticket_agg
WHERE flag_inconsistente = 0
  AND ABS(total - (sub_prod + sub_otros + adj)) >= 0.015
  AND ABS(total - (sub_prod + sub_otros + adj + tax)) >= 0.015
  AND ABS(total - (sub_prod + sub_otros + adj + tax - total_discount)) >= 0.015
  AND ABS(total - (sub_prod + sub_otros + adj - total_discount)) >= 0.015
  AND ABS(total - (sub_prod + sub_otros - item_disc_prod)) >= 0.015
  AND anio_mes BETWEEN '2025-04' AND '2025-10'
  AND n_lineas <= 12
ORDER BY anio_mes, store_id, fecha, check_number
LIMIT 5
""")
display(df_b4a)

print("=== B4b. 5 tickets sin_ecuacion flag=0 de 2026 ===")
df_b4b = spark.sql(base_cte + """
SELECT ticket, store_id, fecha, check_number,
       total, sub_prod, sub_otros, adj, tax, total_discount,
       total - (sub_prod + sub_otros + adj + tax - total_discount) AS d,
       n_lineas
FROM ticket_agg
WHERE flag_inconsistente = 0
  AND ABS(total - (sub_prod + sub_otros + adj)) >= 0.015
  AND ABS(total - (sub_prod + sub_otros + adj + tax)) >= 0.015
  AND ABS(total - (sub_prod + sub_otros + adj + tax - total_discount)) >= 0.015
  AND ABS(total - (sub_prod + sub_otros + adj - total_discount)) >= 0.015
  AND ABS(total - (sub_prod + sub_otros - item_disc_prod)) >= 0.015
  AND anio_mes BETWEEN '2026-03' AND '2026-09'
  AND n_lineas <= 12
ORDER BY anio_mes, store_id, fecha, check_number
LIMIT 5
""")
display(df_b4b)

print("=== B4c. Líneas de los 10 tickets ===")
tickets_b4 = [r.ticket for r in df_b4a.collect()] + [r.ticket for r in df_b4b.collect()]
df_b4c = spark.sql(f"""
SELECT store_id, fecha, check_number, transaction_type, row_type,
       menu_item, pos_code, quantity, menu_price, item_discount,
       total, total_discount, tax
FROM {tbl}
WHERE {ticket_id} IN ({','.join([f"'{t}'" for t in tickets_b4])})
ORDER BY store_id, fecha, check_number, transaction_ts
""")
display(df_b4c)

PARTE A — total_discount contra item_discount

A1 (por mes, 2025-01 a 2026-01): En 2025 (abr-oct), td>0 & id>0 se activa en los mismos tickets (~8%). td>0 & id=0 es insignificante en 2025 (0.01%) pero crece en la transición: 0.12% (nov), 2.26% (dic), 5.67% (ene 2026). td=0 & id>0 siempre es 0%.

A2 (coincidencia y razón): total_discount = item_disc_prod exactamente — razón 1.0000 en todos los percentiles, 99.95-100% de coincidencia. item_disc_all coincide al 100%. No hay factor ni constante: son idénticos. total_discount es la medida única y comparable en toda la serie.

A3 (10 tickets que no coinciden): La discrepancia es siempre por descuentos en líneas delivery_fee. item_disc_all = total_discount en todos los casos. Ej: td=33.40, id_prod=3.57, id_all=33.40 — los 29.83 restantes son item_discount en línea de delivery.

PARTE B — Tickets sin_ecuacion

B1 (por mes): ~0.15% en 2025, sube a ~1% en 2026. Ninguna bandera domina: flag_inconsistente <5%, voids 5-6%, delivery ~0%, lineas_disc 5-41% (bajando), adj 2-3% (2026), max_qty≥10 ~0%. El ~1% está disperso.

B2 (por pop): Drive Thru concentra el problema con 1.45% sin_ecuacion (60K tickets), vs 0.36% Carry Out y 0.33% Dine-In.

B3 (distribución de d): No es redondeo. 2025: mediana d=−$3.99, 65% |d|>$1. 2026: mediana d=−$7.45, 96% |d|>$1, 0% d positiva pequeña. Todas las diferencias son negativas — total es menor que la ecuación dominante.

B4 (inspección visual): Los 10 tickets sin_ecuacion con flag=0 son todos de store 5026, con total=0 pero sub_prod>0. En 2025 tienen tax>0; en 2026 tax=0. Son tickets comped/zeroed — el POS puso total=0 pero dejó las líneas de producto. Acción: excluir con total = 0 AND sub_prod > 0.

Reglas sugeridas para tickets_base:

tiene_promo = total_discount > 0 (medida única, válida en toda la serie)
Bandera de calidad 1: flag_inconsistente (múltiples valores de total/tax/pop/begin_time/total_discount en el ticket)
Bandera de calidad 2: total = 0 AND sub_prod > 0 (tickets comped de store 5026)
item_disc_prod solo sirve en 2025; desde 2026-02 es siempre 0

## Crear la tabla


In [0]:
tbl = "tacobueno.discounts.ventas_clean"
ticket_id = "CONCAT_WS('|', store_id, CAST(fecha AS STRING), check_number)"

print("=== Creando tickets_base ===")
spark.sql(f"""
CREATE OR REPLACE TABLE tacobueno.discounts.tickets_base
USING DELTA
AS
WITH ticket_agg AS (
    SELECT
        {ticket_id} AS ticket_id,
        store_id,
        fecha,
        check_number,
        anio_mes,
        MAX(hora)                                          AS hora,
        MAX(begin_time)                                    AS begin_time,
        MIN(transaction_ts)                                AS ts_ticket,
        MAX(dia_semana)                                    AS dia_semana_orig,
        MAX(LENGTH(check_number))                          AS len_check_number,
        TRIM(MAX(pop))                                     AS pop,
        MAX(total)                                         AS total,
        MAX(total_discount)                                AS total_discount,
        MAX(tax)                                           AS tax,
        COALESCE(SUM(CASE WHEN row_type = 'product' THEN quantity * menu_price ELSE 0 END), 0) AS sub_prod,
        COALESCE(SUM(CASE WHEN row_type IN ('donation','delivery_fee','tip') THEN quantity * menu_price ELSE 0 END), 0) AS sub_otros,
        COALESCE(SUM(CASE WHEN row_type = 'price_adjustment' THEN quantity * menu_price ELSE 0 END), 0) AS adj,
        COALESCE(SUM(CASE WHEN row_type = 'product' THEN item_discount ELSE 0 END), 0) AS item_disc_prod,
        COALESCE(SUM(item_discount), 0)                    AS item_disc_all,
        COALESCE(SUM(CASE WHEN row_type = 'price_adjustment' AND LOWER(regexp_extract(pos_code, '[(]([^)]+)[)]$', 1)) = 'coupon' THEN quantity * menu_price ELSE 0 END), 0) AS adj_coupon,
        COALESCE(SUM(CASE WHEN row_type = 'price_adjustment' AND LOWER(regexp_extract(pos_code, '[(]([^)]+)[)]$', 1)) = 'orderreduction' THEN quantity * menu_price ELSE 0 END), 0) AS adj_orderreduction,
        COALESCE(SUM(CASE WHEN row_type = 'price_adjustment' AND LOWER(regexp_extract(pos_code, '[(]([^)]+)[)]$', 1)) = 'bogo' THEN quantity * menu_price ELSE 0 END), 0) AS adj_bogo,
        SUM(CASE WHEN row_type = 'price_adjustment' AND LOWER(regexp_extract(pos_code, '[(]([^)]+)[)]$', 1)) = 'coupon' THEN 1 ELSE 0 END) AS n_adj_coupon,
        SUM(CASE WHEN row_type = 'price_adjustment' AND LOWER(regexp_extract(pos_code, '[(]([^)]+)[)]$', 1)) = 'orderreduction' THEN 1 ELSE 0 END) AS n_adj_orderreduction,
        SUM(CASE WHEN row_type = 'price_adjustment' AND LOWER(regexp_extract(pos_code, '[(]([^)]+)[)]$', 1)) = 'bogo' THEN 1 ELSE 0 END) AS n_adj_bogo,
        SUM(CASE WHEN row_type = 'price_adjustment' THEN 1 ELSE 0 END) AS n_adj,
        COALESCE(SUM(CASE WHEN row_type = 'product' THEN quantity ELSE 0 END), 0) AS n_items,
        MAX(CASE WHEN row_type = 'product' THEN quantity END) AS max_qty,
        COUNT(*) AS n_lineas,
        SUM(CASE WHEN row_type = 'product' THEN 1 ELSE 0 END) AS n_lineas_prod,
        SUM(CASE WHEN row_type = 'modifier' THEN 1 ELSE 0 END) AS n_modifier,
        SUM(CASE WHEN row_type = 'void' THEN 1 ELSE 0 END) AS n_void,
        SUM(CASE WHEN row_type = 'donation' THEN 1 ELSE 0 END) AS n_donation,
        SUM(CASE WHEN row_type = 'delivery_fee' THEN 1 ELSE 0 END) AS n_delivery_fee,
        SUM(CASE WHEN row_type = 'tip' THEN 1 ELSE 0 END) AS n_tip,
        SUM(CASE WHEN transaction_type = 'Discount' THEN 1 ELSE 0 END) AS n_lineas_discount,
        COALESCE(BOOL_OR(is_partial_day), FALSE) AS is_partial_day,
        COALESCE(BOOL_OR(is_low_coverage_store), FALSE) AS is_low_coverage_store,
        COUNT(DISTINCT total)          AS n_distintos_total,
        COUNT(DISTINCT tax)            AS n_distintos_tax,
        COUNT(DISTINCT pop)            AS n_distintos_pop,
        COUNT(DISTINCT begin_time)     AS n_distintos_begin_time,
        COUNT(DISTINCT total_discount) AS n_distintos_total_discount
    FROM {tbl}
    GROUP BY {ticket_id}, store_id, fecha, check_number, anio_mes
)
SELECT
    ticket_id,
    store_id,
    fecha,
    check_number,
    anio_mes,
    hora,
    begin_time,
    ts_ticket,
    dia_semana_orig,
    DAYOFWEEK(fecha) AS dow_spark,
    len_check_number,
    pop,
    CASE
        WHEN pop = 'Drive Thru' THEN 'drive_thru'
        WHEN pop IN ('To Go', 'Carry Out') THEN 'para_llevar'
        WHEN pop IN ('Inside', 'Dine-In') THEN 'salon'
        WHEN pop IN ('Mobile', 'Online') THEN 'digital_propio'
        WHEN pop IN ('Online: DoorDash', 'Online: UberEats', 'Online: GrubHub', 'Online: Zipline') THEN 'delivery_terceros'
        WHEN pop = '3' THEN 'desconocido'
        ELSE 'otro'
    END AS canal_grupo,
    total,
    total_discount,
    tax,
    sub_prod,
    sub_otros,
    adj,
    item_disc_prod,
    item_disc_all,
    adj_coupon,
    adj_orderreduction,
    adj_bogo,
    adj - adj_coupon - adj_orderreduction - adj_bogo AS adj_otro,
    n_adj_coupon,
    n_adj_orderreduction,
    n_adj_bogo,
    n_adj - n_adj_coupon - n_adj_orderreduction - n_adj_bogo AS n_adj_otro,
    n_adj,
    sub_prod + adj - total_discount AS venta_neta_prod,
    total_discount + ABS(adj) AS monto_desc_total,
    n_items,
    max_qty,
    n_lineas,
    n_lineas_prod,
    n_modifier,
    n_void,
    n_donation,
    n_delivery_fee,
    n_tip,
    n_lineas_discount,
    total_discount > 0 AS promo_td,
    n_adj > 0 AND adj < 0 AS promo_adj,
    (total_discount > 0) OR (n_adj > 0 AND adj < 0) AS tiene_promo,
    total - (sub_prod + sub_otros + adj) AS d_e1,
    total - (sub_prod + sub_otros + adj + tax - total_discount) AS d_e3,
    ABS(total - (sub_prod + sub_otros + adj)) < 0.015 AS cumple_e1,
    ABS(total - (sub_prod + sub_otros + adj + tax - total_discount)) < 0.015 AS cumple_e3,
    (
        ABS(total - (sub_prod + sub_otros + adj)) < 0.015
        OR ABS(total - (sub_prod + sub_otros + adj + tax)) < 0.015
        OR ABS(total - (sub_prod + sub_otros + adj + tax - total_discount)) < 0.015
        OR ABS(total - (sub_prod + sub_otros + adj - total_discount)) < 0.015
        OR ABS(total - (sub_prod + sub_otros - item_disc_prod)) < 0.015
    ) AS cumple_alguna,
    NOT (
        ABS(total - (sub_prod + sub_otros + adj)) < 0.015
        OR ABS(total - (sub_prod + sub_otros + adj + tax)) < 0.015
        OR ABS(total - (sub_prod + sub_otros + adj + tax - total_discount)) < 0.015
        OR ABS(total - (sub_prod + sub_otros + adj - total_discount)) < 0.015
        OR ABS(total - (sub_prod + sub_otros - item_disc_prod)) < 0.015
    ) AS sin_ecuacion,
    CASE WHEN n_distintos_total > 1
              OR n_distintos_tax > 1
              OR n_distintos_pop > 1
              OR n_distintos_begin_time > 1
              OR n_distintos_total_discount > 1
         THEN 1 ELSE 0 END AS flag_inconsistente,
    CASE WHEN total = 0 AND sub_prod > 0 THEN 1 ELSE 0 END AS flag_total_cero,
    is_partial_day,
    is_low_coverage_store,
    CASE
        WHEN fecha < '2025-11-01' THEN 'P1'
        WHEN fecha >= '2025-11-01' AND fecha <= '2026-01-31' THEN 'P2'
        WHEN fecha >= '2026-02-01' THEN 'P3'
    END AS periodo
FROM ticket_agg
""")
print("Tabla creada exitosamente.")
display(spark.sql("SELECT COUNT(*) AS n_filas FROM tacobueno.discounts.tickets_base"))

In [0]:
# ─────────────────────────────────────────────────────────────
# V1. Conteo total
# ─────────────────────────────────────────────────────────────
print("=== V1. Conteo total ===")
df_v1 = spark.sql("""
SELECT COUNT(*)                        AS n_tickets,
       COUNT(DISTINCT ticket_id)       AS n_distinct_ticket_id,
       SUM(n_lineas)                    AS total_lineas
FROM tacobueno.discounts.tickets_base
""")
display(df_v1)

In [0]:
# ─────────────────────────────────────────────────────────────
# V2. Validación de montos contra ventas_clean
# ─────────────────────────────────────────────────────────────
print("=== V2. Validación de montos por anio_mes ===")
df_v2 = spark.sql(f"""
WITH base AS (
    SELECT anio_mes,
           SUM(sub_prod) AS sub_prod_base,
           SUM(adj) AS adj_base,
           SUM(CASE WHEN total_discount > 0 THEN 1 ELSE 0 END) AS tickets_td_pos_base
    FROM tacobueno.discounts.tickets_base
    GROUP BY anio_mes
),
direct AS (
    SELECT anio_mes,
           SUM(CASE WHEN row_type = 'product' THEN quantity * menu_price ELSE 0 END) AS sub_prod_direct,
           SUM(CASE WHEN row_type = 'price_adjustment' THEN quantity * menu_price ELSE 0 END) AS adj_direct
    FROM {tbl}
    GROUP BY anio_mes
),
direct_td AS (
    SELECT anio_mes, COUNT(*) AS tickets_td_pos_direct
    FROM (
        SELECT anio_mes, MAX(total_discount) AS max_td
        FROM {tbl}
        GROUP BY {ticket_id}, anio_mes
        HAVING MAX(total_discount) > 0
    )
    GROUP BY anio_mes
)
SELECT b.anio_mes,
       ROUND(b.sub_prod_base, 2)      AS sub_prod_base,
       ROUND(d.sub_prod_direct, 2)    AS sub_prod_direct,
       ROUND(b.sub_prod_base - d.sub_prod_direct, 2) AS diff_sub_prod,
       ROUND(b.adj_base, 2)           AS adj_base,
       ROUND(d.adj_direct, 2)         AS adj_direct,
       ROUND(b.adj_base - d.adj_direct, 2) AS diff_adj,
       b.tickets_td_pos_base,
       dt.tickets_td_pos_direct,
       b.tickets_td_pos_base - dt.tickets_td_pos_direct AS diff_td_pos
FROM base b
JOIN direct d    ON b.anio_mes = d.anio_mes
JOIN direct_td dt ON b.anio_mes = dt.anio_mes
ORDER BY b.anio_mes
""")
display(df_v2)

In [0]:
# ─────────────────────────────────────────────────────────────
# V3. Resumen de promos por mes
# ─────────────────────────────────────────────────────────────
print("=== V3. Resumen de promos por mes ===")
df_v3 = spark.sql("""
SELECT anio_mes,
       COUNT(*) AS tickets,
       ROUND(SUM(CASE WHEN promo_td THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_promo_td,
       ROUND(SUM(CASE WHEN promo_adj AND NOT promo_td THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_solo_adj,
       ROUND(SUM(CASE WHEN promo_td AND promo_adj THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_ambos,
       ROUND(SUM(CASE WHEN tiene_promo THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_tiene_promo,
       ROUND(AVG(CASE WHEN tiene_promo THEN monto_desc_total END), 2) AS prom_monto_desc,
       ROUND(AVG(CASE WHEN tiene_promo THEN total_discount END), 2) AS prom_td
FROM tacobueno.discounts.tickets_base
GROUP BY anio_mes
ORDER BY anio_mes
""")
display(df_v3)

In [0]:
# ─────────────────────────────────────────────────────────────
# V4. Hipótesis total cero
# ─────────────────────────────────────────────────────────────
print("=== V4a. total cero por mes ===")
df_v4a = spark.sql("""
SELECT anio_mes,
       SUM(flag_total_cero) AS n_total_cero,
       SUM(CASE WHEN flag_total_cero = 1 AND sin_ecuacion THEN 1 ELSE 0 END) AS n_total_cero_y_sin_ec,
       SUM(CASE WHEN flag_total_cero = 1 AND (total_discount > 0 OR n_adj > 0) THEN 1 ELSE 0 END) AS n_total_cero_con_promo,
       SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END) AS n_sin_ecuacion,
       ROUND(SUM(CASE WHEN sin_ecuacion AND flag_total_cero = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END), 0), 2) AS pct_sin_ec_total_cero
FROM tacobueno.discounts.tickets_base
GROUP BY anio_mes
ORDER BY anio_mes
""")
display(df_v4a)

print("=== V4b. Top 15 tiendas por sin_ecuacion ===")
df_v4b = spark.sql("""
SELECT store_id,
       COUNT(*) AS tickets,
       SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END) AS n_sin_ecuacion,
       ROUND(SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_sin_ecuacion,
       ROUND(SUM(CASE WHEN sin_ecuacion AND flag_total_cero = 1 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END), 0), 2) AS pct_sin_ec_total_cero
FROM tacobueno.discounts.tickets_base
GROUP BY store_id
ORDER BY n_sin_ecuacion DESC
LIMIT 15
""")
display(df_v4b)

print("=== V4c. Cruce sin_ecuacion por canal_grupo y periodo ===")
df_v4c = spark.sql("""
SELECT canal_grupo,
       periodo,
       COUNT(*) AS tickets,
       SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END) AS n_sin_ecuacion,
       ROUND(SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_sin_ecuacion
FROM tacobueno.discounts.tickets_base
GROUP BY canal_grupo, periodo
ORDER BY canal_grupo, periodo
""")
display(df_v4c)

In [0]:
# ─────────────────────────────────────────────────────────────
# V5. Cruce dia_semana_orig vs dow_spark
# ─────────────────────────────────────────────────────────────
print("=== V5. Cruce dia_semana_orig vs dow_spark ===")
df_v5 = spark.sql("""
SELECT dia_semana_orig,
       dow_spark,
       COUNT(*) AS n_tickets
FROM tacobueno.discounts.tickets_base
GROUP BY dia_semana_orig, dow_spark
ORDER BY dia_semana_orig, dow_spark
""")
display(df_v5)

In [0]:
# ─────────────────────────────────────────────────────────────
# V6. Fechas de corte de check_number
# ─────────────────────────────────────────────────────────────
print("=== V6. Corte de check_number (2025-10 a 2026-02) ===")
df_v6 = spark.sql("""
SELECT len_check_number,
       MIN(fecha) AS fecha_min,
       MAX(fecha) AS fecha_max,
       COUNT(*) AS n_tickets
FROM tacobueno.discounts.tickets_base
WHERE anio_mes BETWEEN '2025-10' AND '2026-02'
GROUP BY len_check_number
ORDER BY len_check_number
""")
display(df_v6)

In [0]:
# ─────────────────────────────────────────────────────────────
# V7. Resumen por periodo
# ─────────────────────────────────────────────────────────────
print("=== V7. Resumen por periodo ===")
df_v7 = spark.sql("""
SELECT periodo,
       COUNT(*) AS n_tickets,
       COUNT(DISTINCT store_id) AS n_tiendas,
       COUNT(DISTINCT CASE WHEN is_low_coverage_store THEN store_id END) AS n_tiendas_low_coverage
FROM tacobueno.discounts.tickets_base
GROUP BY periodo
ORDER BY periodo
""")
display(df_v7)

## Verify

In [0]:
# ─────────────────────────────────────────────────────────────
# 1. Régimen de total por formato de check_number
# ─────────────────────────────────────────────────────────────
print("=== 1. Régimen de total por formato (2025-10 a 2026-03, tax!=td) ===")
df_1 = spark.sql("""
SELECT CASE WHEN len_check_number IN (6, 14, 15) THEN CAST(len_check_number AS STRING) ELSE 'otros' END AS len_grupo,
       COUNT(*) AS tickets,
       ROUND(SUM(CASE WHEN cumple_e1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_e1,
       ROUND(SUM(CASE WHEN cumple_e3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_e3,
       ROUND(SUM(CASE WHEN cumple_alguna THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_cumple_alguna,
       ROUND(SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_sin_ecuacion,
       ROUND(SUM(CASE WHEN total_discount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_td_pos,
       ROUND(SUM(CASE WHEN item_disc_prod > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_id_pos
FROM tacobueno.discounts.tickets_base
WHERE anio_mes BETWEEN '2025-10' AND '2026-03'
  AND ABS(tax - total_discount) > 0.015
GROUP BY CASE WHEN len_check_number IN (6, 14, 15) THEN CAST(len_check_number AS STRING) ELSE 'otros' END
ORDER BY len_grupo
""")
display(df_1)

In [0]:
# ─────────────────────────────────────────────────────────────
# 2. Migración por tienda: formato y régimen
# ─────────────────────────────────────────────────────────────
print("=== 2a. Migración por tienda: formato ===")
df_2a = spark.sql("""
WITH store_fmt AS (
    SELECT store_id,
           MIN(CASE WHEN len_check_number > 6 THEN fecha END) AS primera_fmt_nuevo,
           MAX(CASE WHEN len_check_number = 6  THEN fecha END) AS ultima_fmt_viejo
    FROM tacobueno.discounts.tickets_base
    GROUP BY store_id
)
SELECT
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_fmt_nuevo), 0.00) AS LONG)) AS p0_primera_nuevo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_fmt_nuevo), 0.25) AS LONG)) AS p25_primera_nuevo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_fmt_nuevo), 0.50) AS LONG)) AS p50_primera_nuevo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_fmt_nuevo), 0.75) AS LONG)) AS p75_primera_nuevo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_fmt_nuevo), 1.00) AS LONG)) AS p100_primera_nuevo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_fmt_viejo), 0.00) AS LONG)) AS p0_ultima_viejo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_fmt_viejo), 0.25) AS LONG)) AS p25_ultima_viejo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_fmt_viejo), 0.50) AS LONG)) AS p50_ultima_viejo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_fmt_viejo), 0.75) AS LONG)) AS p75_ultima_viejo,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_fmt_viejo), 1.00) AS LONG)) AS p100_ultima_viejo,
    SUM(CASE WHEN primera_fmt_nuevo IS NOT NULL AND ultima_fmt_viejo IS NOT NULL
              AND DATEDIFF(ultima_fmt_viejo, primera_fmt_nuevo) >= 0 THEN 1 ELSE 0 END) AS n_convivencia,
    COUNT(*) AS n_tiendas
FROM store_fmt
""")
display(df_2a)

print("=== 2b. Migración por tienda: régimen de total ===")
df_2b = spark.sql("""
WITH store_eq AS (
    SELECT store_id,
           MIN(CASE WHEN cumple_e3 AND NOT cumple_e1 AND tax > 0 THEN fecha END) AS primera_e3,
           MAX(CASE WHEN cumple_e1 AND NOT cumple_e3 AND tax > 0 THEN fecha END) AS ultima_e1
    FROM tacobueno.discounts.tickets_base
    GROUP BY store_id
)
SELECT
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_e3), 0.00) AS LONG)) AS p0_primera_e3,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_e3), 0.25) AS LONG)) AS p25_primera_e3,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_e3), 0.50) AS LONG)) AS p50_primera_e3,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_e3), 0.75) AS LONG)) AS p75_primera_e3,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(primera_e3), 1.00) AS LONG)) AS p100_primera_e3,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_e1), 0.00) AS LONG)) AS p0_ultima_e1,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_e1), 0.25) AS LONG)) AS p25_ultima_e1,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_e1), 0.50) AS LONG)) AS p50_ultima_e1,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_e1), 0.75) AS LONG)) AS p75_ultima_e1,
    DATE_FROM_UNIX_DATE(CAST(percentile_approx(UNIX_DATE(ultima_e1), 1.00) AS LONG)) AS p100_ultima_e1
FROM store_eq
""")
display(df_2b)

print("=== 2c. Diferencia en días: primera_e3 vs primera_fmt ===")
df_2c = spark.sql("""
WITH store_eq AS (
    SELECT store_id,
           MIN(CASE WHEN cumple_e3 AND NOT cumple_e1 AND tax > 0 THEN fecha END) AS primera_e3
    FROM tacobueno.discounts.tickets_base
    GROUP BY store_id
),
store_fmt AS (
    SELECT store_id,
           MIN(CASE WHEN len_check_number > 6 THEN fecha END) AS primera_fmt
    FROM tacobueno.discounts.tickets_base
    GROUP BY store_id
)
SELECT
    percentile_approx(DATEDIFF(e.primera_e3, f.primera_fmt), 0.00) AS p0,
    percentile_approx(DATEDIFF(e.primera_e3, f.primera_fmt), 0.25) AS p25,
    percentile_approx(DATEDIFF(e.primera_e3, f.primera_fmt), 0.50) AS p50,
    percentile_approx(DATEDIFF(e.primera_e3, f.primera_fmt), 0.75) AS p75,
    percentile_approx(DATEDIFF(e.primera_e3, f.primera_fmt), 1.00) AS p100,
    COUNT(*) AS n_tiendas
FROM store_eq e
JOIN store_fmt f ON e.store_id = f.store_id
WHERE e.primera_e3 IS NOT NULL AND f.primera_fmt IS NOT NULL
""")
display(df_2c)

In [0]:
# ─────────────────────────────────────────────────────────────
# 3. Formato de 15 dígitos
# ─────────────────────────────────────────────────────────────
print("=== 3. Formato 15 dígitos (desde 2025-12) ===")
df_3 = spark.sql("""
WITH base AS (
    SELECT anio_mes,
           SUM(CASE WHEN len_check_number = 14 THEN 1 ELSE 0 END) AS tickets_14,
           SUM(CASE WHEN len_check_number = 15 THEN 1 ELSE 0 END) AS tickets_15,
           COUNT(DISTINCT CASE WHEN len_check_number = 14 THEN store_id END) AS tiendas_14,
           COUNT(DISTINCT CASE WHEN len_check_number = 15 THEN store_id END) AS tiendas_15
    FROM tacobueno.discounts.tickets_base
    WHERE anio_mes >= '2025-12' AND len_check_number IN (14, 15)
    GROUP BY anio_mes
),
store_fecha_fmt AS (
    SELECT anio_mes, store_id, fecha,
           MAX(CASE WHEN len_check_number = 14 THEN 1 ELSE 0 END) AS has_14,
           MAX(CASE WHEN len_check_number = 15 THEN 1 ELSE 0 END) AS has_15
    FROM tacobueno.discounts.tickets_base
    WHERE anio_mes >= '2025-12' AND len_check_number IN (14, 15)
    GROUP BY anio_mes, store_id, fecha
),
ambos AS (
    SELECT anio_mes, COUNT(*) AS store_fechas_ambos
    FROM store_fecha_fmt
    WHERE has_14 = 1 AND has_15 = 1
    GROUP BY anio_mes
)
SELECT b.anio_mes,
       b.tickets_14,
       b.tickets_15,
       b.tiendas_14,
       b.tiendas_15,
       COALESCE(a.store_fechas_ambos, 0) AS store_fechas_ambos
FROM base b
LEFT JOIN ambos a ON b.anio_mes = a.anio_mes
ORDER BY b.anio_mes
""")
display(df_3)

In [0]:
# ─────────────────────────────────────────────────────────────
# 4. Sin ecuación que no son total cero
# ─────────────────────────────────────────────────────────────
print("=== 4a. Sin ecuacion (no total cero) por periodo ===")
df_4a = spark.sql("""
SELECT periodo,
       COUNT(*) AS n_tickets,
       ROUND(SUM(CASE WHEN n_void > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_n_void,
       ROUND(SUM(CASE WHEN n_lineas_discount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_lineas_disc,
       ROUND(SUM(CASE WHEN n_adj > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_adj,
       ROUND(percentile_approx(CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END, 0.05), 4) AS p5_d,
       ROUND(percentile_approx(CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END, 0.25), 4) AS p25_d,
       ROUND(percentile_approx(CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END, 0.50), 4) AS p50_d,
       ROUND(percentile_approx(CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END, 0.75), 4) AS p75_d,
       ROUND(percentile_approx(CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END, 0.95), 4) AS p95_d,
       ROUND(percentile_approx(CASE WHEN sub_prod > 0 THEN CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END / sub_prod END, 0.05), 4) AS p5_d_sp,
       ROUND(percentile_approx(CASE WHEN sub_prod > 0 THEN CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END / sub_prod END, 0.25), 4) AS p25_d_sp,
       ROUND(percentile_approx(CASE WHEN sub_prod > 0 THEN CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END / sub_prod END, 0.50), 4) AS p50_d_sp,
       ROUND(percentile_approx(CASE WHEN sub_prod > 0 THEN CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END / sub_prod END, 0.75), 4) AS p75_d_sp,
       ROUND(percentile_approx(CASE WHEN sub_prod > 0 THEN CASE WHEN periodo = 'P1' THEN d_e1 ELSE d_e3 END / sub_prod END, 0.95), 4) AS p95_d_sp
FROM tacobueno.discounts.tickets_base
WHERE sin_ecuacion = true AND flag_total_cero = 0
GROUP BY periodo
ORDER BY periodo
""")
display(df_4a)

print("=== 4b. Cruce por canal_grupo y periodo ===")
df_4b = spark.sql("""
SELECT canal_grupo,
       periodo,
       COUNT(*) AS n_total,
       SUM(CASE WHEN sin_ecuacion AND flag_total_cero = 0 THEN 1 ELSE 0 END) AS n_sin_ec_no_cero,
       ROUND(SUM(CASE WHEN sin_ecuacion AND flag_total_cero = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS pct_sin_ec_no_cero
FROM tacobueno.discounts.tickets_base
GROUP BY canal_grupo, periodo
ORDER BY canal_grupo, periodo
""")
display(df_4b)

In [0]:
# ─────────────────────────────────────────────────────────────
# 5. Concentración por tienda
# ─────────────────────────────────────────────────────────────
print("=== 5a. Top 15 tiendas por sin_ecuacion ===")
df_5a = spark.sql("""
WITH store_se AS (
    SELECT store_id,
           COUNT(*) AS tickets,
           SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END) AS n_sin_ecuacion
    FROM tacobueno.discounts.tickets_base
    GROUP BY store_id
),
total_se AS (
    SELECT SUM(n_sin_ecuacion) AS total_sin_ecuacion FROM store_se
)
SELECT s.store_id,
       s.tickets,
       s.n_sin_ecuacion,
       ROUND(s.n_sin_ecuacion * 100.0 / s.tickets, 2) AS pct_store,
       ROUND(s.n_sin_ecuacion * 100.0 / t.total_sin_ecuacion, 2) AS pct_total,
       ROUND(SUM(s.n_sin_ecuacion) OVER (ORDER BY s.n_sin_ecuacion DESC) * 100.0 / t.total_sin_ecuacion, 2) AS pct_acum
FROM store_se s
CROSS JOIN total_se t
ORDER BY s.n_sin_ecuacion DESC
LIMIT 15
""")
display(df_5a)

print("=== 5b. Tiendas con >3% sin_ecuacion ===")
df_5b = spark.sql("""
SELECT COUNT(*) AS n_tiendas_3pct
FROM (
    SELECT store_id,
           SUM(CASE WHEN sin_ecuacion THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_sin_ecuacion
    FROM tacobueno.discounts.tickets_base
    GROUP BY store_id
)
WHERE pct_sin_ecuacion > 3.0
""")
display(df_5b)

In [0]:
# ─────────────────────────────────────────────────────────────
# 6. Día de la semana
# ─────────────────────────────────────────────────────────────
print("=== 6. Día de la semana ===")
df_6 = spark.sql("""
SELECT dow_spark,
       MAX(date_format(fecha, 'E')) AS nombre_dia,
       COUNT(*) AS n_tickets
FROM tacobueno.discounts.tickets_base
GROUP BY dow_spark
ORDER BY dow_spark
""")
display(df_6)